# Etsy Product Image Binary Classification

**Goal:** Maximise F1 score to **0.85–0.90** using:
- Transfer learning (EfficientNet-B0 + Vision Transformer)
- Transformer Fusion via concatenated embeddings
- XGBoost stacking ensemble
- Test Time Augmentation (TTA)
- Threshold optimisation
- Grad-CAM interpretability

---

## Cell 0 — Setup

Install required packages, import all libraries, set global random seeds for full reproducibility, detect CUDA, enable mixed-precision (AMP), and define memory-cleanup helpers.  
We also include a **robust filename-cleaning function** that renames any files containing brackets `()`, spaces, commas, or duplicate symbols so that `torchvision.datasets.ImageFolder` can load them without errors.

In [ ]:
# ── 0.1  Install extra packages not pre-installed in Colab ──────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "xgboost", "grad-cam"], check=False)

# ── 0.2  Standard imports ────────────────────────────────────────────────────
import os, re, gc, shutil, random, warnings, copy
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder

import timm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.contingency_tables import mcnemar

import xgboost as xgb

warnings.filterwarnings("ignore")

# ── 0.3  Reproducibility ─────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Fix seeds for Python, NumPy and PyTorch (CPU + GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── 0.4  Device & mixed-precision ────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()          # only enable AMP with CUDA
scaler  = GradScaler(enabled=USE_AMP)
print(f"Device : {DEVICE}  |  AMP : {USE_AMP}")

# ── 0.5  Memory-cleanup helpers ───────────────────────────────────────────────
def free_memory() -> None:
    """Release PyTorch GPU cache and run Python garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ── 0.6  Dataset paths ────────────────────────────────────────────────────────
DATA_ROOT  = Path("/content/dataset")
TRAIN_DIR  = DATA_ROOT / "train"
TEST_DIR   = DATA_ROOT / "test"

# ── 0.7  Robust filename-cleaning function ────────────────────────────────────
def clean_filenames(root: Path) -> int:
    """
    Walk *root* recursively and rename every file that contains:
      - round brackets  ( )
      - spaces
      - commas
      - duplicate dots or consecutive underscores

    Returns the number of files renamed.
    """
    renamed = 0
    for fpath in sorted(root.rglob("*")):
        if not fpath.is_file():
            continue
        stem, suffix = fpath.stem, fpath.suffix
        # Replace problematic characters
        new_stem = re.sub(r"[()\s,]+", "_", stem)   # brackets/spaces/commas → _
        new_stem = re.sub(r"_+",        "_", new_stem)  # collapse duplicate _
        new_stem = new_stem.strip("_")                   # trim leading/trailing _
        new_name = new_stem + suffix
        if new_name != fpath.name:
            dest = fpath.parent / new_name
            # Avoid collision: append counter if dest already exists
            counter = 1
            while dest.exists():
                dest = fpath.parent / f"{new_stem}_{counter}{suffix}"
                counter += 1
            fpath.rename(dest)
            renamed += 1
    return renamed

if DATA_ROOT.exists():
    n = clean_filenames(DATA_ROOT)
    print(f"Cleaned {n} filenames.")
else:
    print("WARNING: /content/dataset not found — skipping filename cleaning.")

# ── 0.8  Dataset class counts ─────────────────────────────────────────────────
def count_classes(split_dir: Path) -> dict:
    counts = {}
    if split_dir.exists():
        for cls in sorted(split_dir.iterdir()):
            if cls.is_dir():
                counts[cls.name] = len(list(cls.glob("*")))
    return counts

train_counts = count_classes(TRAIN_DIR)
test_counts  = count_classes(TEST_DIR)
print("Train class counts:", train_counts)
print("Test  class counts:", test_counts)

## Cell 1 — Data Pipeline

We define **aggressive training augmentations** (random flips, rotation, colour jitter, affine, grayscale) and light validation/test transforms (just resize + normalize).  
A **stratified train/validation split** (80 / 20) is created so that both splits have the same class ratio.  
DataLoaders use `pin_memory=True` and `num_workers=2` for fast GPU feeding.

In [ ]:
# ── 1.1  Image size & hyper-params ───────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── 1.2  Transforms ───────────────────────────────────────────────────────────
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── 1.3  Load full training set (no transform yet — set after splitting) ──────
full_train_dataset = ImageFolder(root=str(TRAIN_DIR))
test_dataset       = ImageFolder(root=str(TEST_DIR), transform=val_transform)

print("Classes:", full_train_dataset.classes)
labels = [s[1] for s in full_train_dataset.samples]

# ── 1.4  Stratified split (80 % train / 20 % val) ────────────────────────────
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

# Build Subset datasets with different transforms using a wrapper
class TransformSubset(torch.utils.data.Dataset):
    """Wraps a Subset and applies a transform on the fly."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

    @property
    def targets(self):
        return [self.subset.dataset.samples[i][1] for i in self.subset.indices]

# The base dataset returns PIL images when no transform is set
full_train_dataset.transform = None      # keep PIL images

train_subset = TransformSubset(Subset(full_train_dataset, train_idx), train_transform)
val_subset   = TransformSubset(Subset(full_train_dataset, val_idx),   val_transform)

# ── 1.5  DataLoaders ──────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

print(f"Train samples : {len(train_subset):>5}")
print(f"Val   samples : {len(val_subset):>5}")
print(f"Test  samples : {len(test_dataset):>5}")
print(f"Val class dist: {Counter(val_subset.targets)}")

# ── 1.6  Visualise a batch ────────────────────────────────────────────────────
imgs, lbls = next(iter(train_loader))
grid = torchvision.utils.make_grid(
    imgs[:16].cpu(),
    nrow=8, normalize=True, value_range=(-1, 1)
)
plt.figure(figsize=(14, 4))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("Sample training batch (augmented)")
plt.axis("off")
plt.tight_layout()
plt.show()

## Cell 2 — Fine-Tuned EfficientNet-B0

We load ImageNet-pretrained **EfficientNet-B0** from `torchvision` and replace its classifier with a custom binary head (Linear → ReLU → Dropout → Linear).  

Training proceeds in **two phases**:
| Phase | Layers trained | Epochs | LR |
|-------|---------------|--------|----|
| 1 | Head only (backbone frozen) | 3 | 1e-3 |
| 2 | Head + top MBConv blocks (features[6:]) | 5 | 5e-5 |

Loss: `BCEWithLogitsLoss` · Optimizer: `AdamW` · Scheduler: cosine annealing.  
**Early stopping** monitors validation F1 (patience = 3) and **best checkpoint** is saved to disk.

In [ ]:
# ── 2.1  Build EfficientNet-B0 with custom binary head ───────────────────────
def build_efficientnet() -> nn.Module:
    model = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1")
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, 1),
    )
    return model

eff_model = build_efficientnet().to(DEVICE)

# ── 2.2  Training utilities ───────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, amp_scaler):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        amp_scaler.scale(loss).backward()
        amp_scaler.step(optimizer)
        amp_scaler.update()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    """Returns (avg_loss, f1, roc_auc, all_probs, all_labels)."""
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss, all_probs, all_labels = 0.0, [], []
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels_t = labels.float().unsqueeze(1).to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels_t)
        total_loss += loss.item() * imgs.size(0)
        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds      = (all_probs >= 0.5).astype(int)
    f1         = f1_score(all_labels, preds, zero_division=0)
    auc        = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0
    return total_loss / len(loader.dataset), f1, auc, all_probs, all_labels


def run_training_phase(
    model, train_loader, val_loader,
    epochs, lr, phase_name,
    ckpt_path="best_efficientnet.pth",
    patience=3
):
    """Generic training loop with early stopping and best-checkpoint saving."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    amp_scaler = GradScaler(enabled=USE_AMP)

    best_f1, no_improve = 0.0, 0
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_auc": []}

    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, amp_scaler)
        vl_loss, vl_f1, vl_auc, _, _ = evaluate(model, val_loader)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["val_f1"].append(vl_f1)
        history["val_auc"].append(vl_auc)

        print(f"[{phase_name}] Epoch {epoch:02d}/{epochs}  "
              f"train_loss={tr_loss:.4f}  val_loss={vl_loss:.4f}  "
              f"val_F1={vl_f1:.4f}  val_AUC={vl_auc:.4f}")

        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch} (best val_F1={best_f1:.4f})")
                break

    print(f"  → Best val F1 after {phase_name}: {best_f1:.4f}")
    return history, best_f1

# ── 2.3  Phase 1: freeze backbone, train head ─────────────────────────────────
for param in eff_model.parameters():
    param.requires_grad = False
for param in eff_model.classifier.parameters():
    param.requires_grad = True

print("=" * 60)
print("PHASE 1 — head-only training (3 epochs)")
print("=" * 60)
hist_p1, f1_p1 = run_training_phase(
    eff_model, train_loader, val_loader,
    epochs=3, lr=1e-3, phase_name="Phase-1",
    ckpt_path="best_efficientnet.pth"
)

# ── 2.4  Phase 2: unfreeze top MBConv blocks ──────────────────────────────────
# EfficientNet-B0 backbone is model.features; unfreeze blocks 6 and 7 + head
for param in eff_model.parameters():
    param.requires_grad = False
for block in eff_model.features[6:]:
    for param in block.parameters():
        param.requires_grad = True
for param in eff_model.classifier.parameters():
    param.requires_grad = True

print("\n" + "=" * 60)
print("PHASE 2 — top MBConv + head fine-tuning (5 epochs)")
print("=" * 60)
hist_p2, f1_p2 = run_training_phase(
    eff_model, train_loader, val_loader,
    epochs=5, lr=5e-5, phase_name="Phase-2",
    ckpt_path="best_efficientnet.pth"
)

# ── 2.5  Load best weights and final evaluation ───────────────────────────────
eff_model.load_state_dict(torch.load("best_efficientnet.pth", map_location=DEVICE))
_, eff_val_f1, eff_val_auc, eff_val_probs, val_labels = evaluate(eff_model, val_loader)
print(f"\nEfficientNet-B0 final  val_F1={eff_val_f1:.4f}  val_AUC={eff_val_auc:.4f}")
print(f"Expected F1 after EfficientNet: ~{eff_val_f1:.3f}")

free_memory()

# ── 2.6  Plot training curves ─────────────────────────────────────────────────
all_history = {
    k: hist_p1[k] + hist_p2[k] for k in hist_p1
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(all_history["train_loss"], label="Train")
axes[0].plot(all_history["val_loss"],   label="Val")
axes[0].set_title("EfficientNet — Loss")
axes[0].legend()
axes[1].plot(all_history["val_f1"])
axes[1].set_title("EfficientNet — Val F1")
axes[2].plot(all_history["val_auc"])
axes[2].set_title("EfficientNet — Val AUC")
for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

## Cell 3 — Vision Transformer (ViT-B/16)

We load **`vit_base_patch16_224`** from `timm` (ImageNet-pretrained).  
The `head` is replaced with the same `Linear(768, 256) → ReLU → Dropout → Linear(256, 1)` architecture.  
Most transformer blocks are frozen; only the **last two encoder blocks** and the new head are trained.  
Training uses a low learning rate (1e-4) with cosine annealing and the same early-stopping strategy.

In [ ]:
# ── 3.1  Build ViT-B/16 with custom binary head ───────────────────────────────
def build_vit() -> nn.Module:
    model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
    in_features = model.embed_dim          # 768 for ViT-B
    model.head = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, 1),
    )
    return model

vit_model = build_vit().to(DEVICE)

# ── 3.2  Freeze all, then unfreeze last 2 transformer blocks ──────────────────
for param in vit_model.parameters():
    param.requires_grad = False

# ViT blocks are in vit_model.blocks (list of transformer encoder blocks)
for block in vit_model.blocks[-2:]:
    for param in block.parameters():
        param.requires_grad = True

# Always unfreeze the norm layer and head
for param in vit_model.norm.parameters():
    param.requires_grad = True
for param in vit_model.head.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in vit_model.parameters())
print(f"ViT trainable params: {trainable:,} / {total:,} "
      f"({100 * trainable / total:.1f} %)")

# ── 3.3  Train ViT ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("ViT-B/16 fine-tuning (6 epochs, lr=1e-4)")
print("=" * 60)
vit_hist, vit_best_f1 = run_training_phase(
    vit_model, train_loader, val_loader,
    epochs=6, lr=1e-4, phase_name="ViT",
    ckpt_path="best_vit.pth",
    patience=3
)

# ── 3.4  Load best weights and evaluate ───────────────────────────────────────
vit_model.load_state_dict(torch.load("best_vit.pth", map_location=DEVICE))
_, vit_val_f1, vit_val_auc, vit_val_probs, _ = evaluate(vit_model, val_loader)
print(f"\nViT-B/16 final  val_F1={vit_val_f1:.4f}  val_AUC={vit_val_auc:.4f}")
print(f"Expected F1 improvement after ViT ensemble: ~{max(eff_val_f1, vit_val_f1):.3f}+")

free_memory()

# ── 3.5  Plot ViT training curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(vit_hist["train_loss"], label="Train")
axes[0].plot(vit_hist["val_loss"],   label="Val")
axes[0].set_title("ViT — Loss")
axes[0].legend()
axes[1].plot(vit_hist["val_f1"])
axes[1].set_title("ViT — Val F1")
axes[2].plot(vit_hist["val_auc"])
axes[2].set_title("ViT — Val AUC")
for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

## Cell 4 — Embedding Extraction + Meta-Learner Benchmarking

Instead of stacking scalar probabilities, we extract the **256-dimensional penultimate embeddings** from each deep model using **forward hooks** and immediately move them to CPU numpy arrays to minimise GPU memory pressure.  
The two embedding vectors are concatenated into a **512-dimensional representation fusion**:

```
EfficientNet  →  256-d embedding  ─┐
                                    ├── [512-d] → Meta-Learner → P(class=1)
ViT           →  256-d embedding  ─┘
```

We benchmark **three meta-learners** on the same 512-d features:

| # | Meta-Learner | Key settings |
|---|-------------|-------------|
| 1 | **XGBoost** (baseline) | max_depth=4, lr=0.03, n_estimators=300, subsample/colsample=0.9 |
| 2 | **CatBoost** | iterations=500, depth=6, lr=0.03, eval_metric=F1, auto_class_weights=Balanced |
| 3 | **TabPFN** | default settings (in-context learning transformer, no training required) |

> **No validation leakage:** all meta-learners are fitted on **train embeddings only**.  
> Validation embeddings are used exclusively for evaluation and threshold selection.  
> The test set is never seen during any fitting or tuning step.

Each learner undergoes **per-learner threshold optimisation** (sweep 0.20–0.80 on validation F1).  
The best learner by validation F1 is used for all downstream TTA and test evaluation.

In [ ]:
# ── 4.1  Install extra meta-learner packages ─────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "catboost"], check=False)
# TabPFN is attempted separately — may not be available on all Colab tiers
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "tabpfn"], check=False)
    TABPFN_AVAILABLE = True
except Exception:
    TABPFN_AVAILABLE = False

# ── 4.2  Hook-based embedding extractor ──────────────────────────────────────
class EmbeddingExtractor:
    """
    Attaches a forward hook to *layer* and captures its output immediately
    on the CPU as a numpy array to avoid holding GPU tensors between batches.
    """
    def __init__(self, model: nn.Module, layer: nn.Module):
        self.embedding = None
        self._hook     = layer.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, input, output):
        # Detach and move to CPU immediately to free GPU memory
        self.embedding = output.detach().cpu()

    def remove(self):
        self._hook.remove()

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.remove()


@torch.no_grad()
def extract_embeddings(model, loader, penultimate_layer):
    """
    Run *loader* through *model* and return
    (embeddings [N, D] numpy, labels [N] numpy, probs [N] numpy).
    Embeddings are moved to CPU numpy immediately after each batch.
    """
    model.eval()
    extractor = EmbeddingExtractor(model, penultimate_layer)
    all_embs, all_labels, all_probs = [], [], []

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)      # forward hook fires here
        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        all_embs.append(extractor.embedding.numpy())   # already on CPU
        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())

    extractor.remove()
    free_memory()   # clear any GPU residuals
    return (
        np.vstack(all_embs),
        np.array(all_labels),
        np.array(all_probs)
    )


# ── 4.3  Identify penultimate layers ─────────────────────────────────────────
# EfficientNet classifier = [Linear(1280,256), ReLU, Dropout, Linear(256,1)]
#   hook target = classifier[1] (the ReLU module; hook captures its 256-d output)
eff_penultimate = eff_model.classifier[1]

# ViT head = [Linear(768,256), ReLU, Dropout, Linear(256,1)]
#   hook target = head[1]
vit_penultimate = vit_model.head[1]

# ── 4.4  Extract embeddings (train / val / test) ──────────────────────────────
print("Extracting EfficientNet embeddings…")
eff_val_emb,   val_labels_emb,   _ = extract_embeddings(eff_model, val_loader,   eff_penultimate)
eff_test_emb,  test_labels_emb,  _ = extract_embeddings(eff_model, test_loader,  eff_penultimate)
eff_train_emb, train_labels_emb, _ = extract_embeddings(eff_model, train_loader, eff_penultimate)

print("Extracting ViT embeddings…")
vit_val_emb,   _, _ = extract_embeddings(vit_model, val_loader,   vit_penultimate)
vit_test_emb,  _, _ = extract_embeddings(vit_model, test_loader,  vit_penultimate)
vit_train_emb, _, _ = extract_embeddings(vit_model, train_loader, vit_penultimate)

# ── 4.5  Concatenate → 512-d (representation fusion) ─────────────────────────
X_train_stack = np.hstack([eff_train_emb, vit_train_emb])   # (N_train, 512)
X_val_stack   = np.hstack([eff_val_emb,   vit_val_emb])     # (N_val,   512)
X_test_stack  = np.hstack([eff_test_emb,  vit_test_emb])    # (N_test,  512)

print(f"Stacked shapes — train: {X_train_stack.shape}, "
      f"val: {X_val_stack.shape}, test: {X_test_stack.shape}")

# Standardise features (fitted on train only — no leakage)
feat_scaler   = StandardScaler()
X_train_stack = feat_scaler.fit_transform(X_train_stack)
X_val_stack   = feat_scaler.transform(X_val_stack)
X_test_stack  = feat_scaler.transform(X_test_stack)

free_memory()

# ─────────────────────────────────────────────────────────────────────────────
# PART B — TRAIN THREE META-LEARNERS
# All learners are fitted on train embeddings only (no leakage).
# ─────────────────────────────────────────────────────────────────────────────

# ── 4.6  XGBoost baseline ────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("META-LEARNER 1 — XGBoost")
print("=" * 55)
xgb_meta = xgb.XGBClassifier(
    max_depth=4,
    learning_rate=0.03,
    n_estimators=300,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
xgb_meta.fit(
    X_train_stack, train_labels_emb,
    eval_set=[(X_val_stack, val_labels_emb)],
    verbose=50,
)
xgb_val_probs  = xgb_meta.predict_proba(X_val_stack)[:, 1]
xgb_test_probs = xgb_meta.predict_proba(X_test_stack)[:, 1]
print(f"XGBoost raw val AUC = {roc_auc_score(val_labels_emb, xgb_val_probs):.4f}")

# ── 4.7  CatBoost ────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("META-LEARNER 2 — CatBoost")
print("=" * 55)
from catboost import CatBoostClassifier

cat_meta = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.03,
    eval_metric="F1",
    auto_class_weights="Balanced",
    early_stopping_rounds=50,
    random_seed=SEED,
    verbose=50,
    task_type="GPU" if torch.cuda.is_available() else "CPU",
)
cat_meta.fit(
    X_train_stack, train_labels_emb,
    eval_set=(X_val_stack, val_labels_emb),
)
cat_val_probs  = cat_meta.predict_proba(X_val_stack)[:, 1]
cat_test_probs = cat_meta.predict_proba(X_test_stack)[:, 1]
print(f"CatBoost raw val AUC = {roc_auc_score(val_labels_emb, cat_val_probs):.4f}")

# ── 4.8  TabPFN ──────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("META-LEARNER 3 — TabPFN")
print("=" * 55)
tabpfn_meta       = None
tabpfn_val_probs  = None
tabpfn_test_probs = None

try:
    from tabpfn import TabPFNClassifier
    tabpfn_meta = TabPFNClassifier(device="cpu", N_ensemble_configurations=16)
    # TabPFN works best with <= 1000 training samples; subsample if necessary
    N_TABPFN = min(1000, len(X_train_stack))
    rng = np.random.default_rng(SEED)
    idx_sub = rng.choice(len(X_train_stack), size=N_TABPFN, replace=False)
    tabpfn_meta.fit(X_train_stack[idx_sub], train_labels_emb[idx_sub])
    tabpfn_val_probs  = tabpfn_meta.predict_proba(X_val_stack)[:, 1]
    tabpfn_test_probs = tabpfn_meta.predict_proba(X_test_stack)[:, 1]
    print(f"TabPFN raw val AUC = {roc_auc_score(val_labels_emb, tabpfn_val_probs):.4f}")
    TABPFN_AVAILABLE = True
except Exception as exc:
    print(f"TabPFN unavailable or failed ({exc}). It will be excluded from benchmarks.")
    TABPFN_AVAILABLE = False

# ─────────────────────────────────────────────────────────────────────────────
# PART C — PER-LEARNER THRESHOLD OPTIMISATION
# Threshold sweep on VALIDATION set only. Test set never touched here.
# ─────────────────────────────────────────────────────────────────────────────
def optimise_threshold(val_probs, val_labels, low=0.20, high=0.80, steps=61):
    """Return (best_threshold, best_f1) over a uniform grid."""
    thresholds = np.linspace(low, high, steps)
    f1s = [f1_score(val_labels, (val_probs >= t).astype(int), zero_division=0)
           for t in thresholds]
    best_idx = int(np.argmax(f1s))
    return float(thresholds[best_idx]), float(f1s[best_idx])

xgb_best_thr,    xgb_best_val_f1    = optimise_threshold(xgb_val_probs,    val_labels_emb)
cat_best_thr,    cat_best_val_f1    = optimise_threshold(cat_val_probs,    val_labels_emb)

print(f"\nXGBoost  best thr={xgb_best_thr:.2f}  val_F1={xgb_best_val_f1:.4f}")
print(f"CatBoost best thr={cat_best_thr:.2f}  val_F1={cat_best_val_f1:.4f}")

if TABPFN_AVAILABLE:
    tabpfn_best_thr, tabpfn_best_val_f1 = optimise_threshold(tabpfn_val_probs, val_labels_emb)
    print(f"TabPFN   best thr={tabpfn_best_thr:.2f}  val_F1={tabpfn_best_val_f1:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# PART D — BENCHMARK OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import average_precision_score

def learner_metrics(name, val_probs, val_labels, thr):
    preds = (val_probs >= thr).astype(int)
    return {
        "model":         name,
        "val_F1":        round(f1_score(val_labels, preds, zero_division=0),           4),
        "val_ROC_AUC":   round(roc_auc_score(val_labels, val_probs),                   4),
        "val_PR_AUC":    round(average_precision_score(val_labels, val_probs),          4),
        "val_Precision": round(float(np.sum((preds == 1) & (val_labels == 1)) /
                               max(np.sum(preds == 1), 1)),                             4),
        "val_Recall":    round(float(np.sum((preds == 1) & (val_labels == 1)) /
                               max(np.sum(val_labels == 1), 1)),                        4),
        "best_threshold": thr,
    }

rows = [
    learner_metrics("XGBoost",  xgb_val_probs,  val_labels_emb, xgb_best_thr),
    learner_metrics("CatBoost", cat_val_probs,  val_labels_emb, cat_best_thr),
]
if TABPFN_AVAILABLE:
    rows.append(learner_metrics("TabPFN", tabpfn_val_probs, val_labels_emb, tabpfn_best_thr))

benchmark_df = pd.DataFrame(rows).sort_values("val_F1", ascending=False).reset_index(drop=True)
print("\n── Benchmark Results (sorted by val F1) ──")
print(benchmark_df.to_string(index=False))

# ── Bar plot: F1 comparison ────────────────────────────────────────────────
palette = ["#4C72B0", "#DD8452", "#55A868"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(benchmark_df["model"], benchmark_df["val_F1"],
       color=palette[:len(benchmark_df)], edgecolor="black")
ax.set_ylabel("Validation F1")
ax.set_title("Meta-Learner F1 Comparison")
ax.set_ylim(0, 1)
for i, v in enumerate(benchmark_df["val_F1"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

# ── ROC curves for all learners ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
all_val_probs = {"XGBoost": xgb_val_probs, "CatBoost": cat_val_probs}
if TABPFN_AVAILABLE:
    all_val_probs["TabPFN"] = tabpfn_val_probs
colors_roc = ["#4C72B0", "#DD8452", "#55A868"]
for (name, probs), col in zip(all_val_probs.items(), colors_roc):
    fpr, tpr, _ = roc_curve(val_labels_emb, probs)
    auc = roc_auc_score(val_labels_emb, probs)
    ax.plot(fpr, tpr, color=col, lw=2, label=f"{name} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Meta-Learner Comparison")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ── PR curves for all learners ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
for (name, probs), col in zip(all_val_probs.items(), colors_roc):
    prec_c, rec_c, _ = precision_recall_curve(val_labels_emb, probs)
    pr_auc = average_precision_score(val_labels_emb, probs)
    ax.plot(rec_c, prec_c, color=col, lw=2, label=f"{name} (AP={pr_auc:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Meta-Learner Comparison")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# ── Identify best meta-learner ─────────────────────────────────────────────
best_meta_name = benchmark_df.iloc[0]["model"]
best_meta_thr  = benchmark_df.iloc[0]["best_threshold"]
print(f"\n★  Best meta-learner: {best_meta_name} "
      f"(val_F1={benchmark_df.iloc[0]['val_F1']:.4f}, "
      f"threshold={best_meta_thr:.2f})")

# Map name → fitted model object and pre-computed test probabilities
_meta_map = {
    "XGBoost":  (xgb_meta,    xgb_val_probs,    xgb_test_probs,    xgb_best_thr),
    "CatBoost": (cat_meta,    cat_val_probs,    cat_test_probs,    cat_best_thr),
}
if TABPFN_AVAILABLE:
    _meta_map["TabPFN"] = (tabpfn_meta, tabpfn_val_probs, tabpfn_test_probs, tabpfn_best_thr)

best_meta, best_meta_val_probs, best_meta_test_probs, best_meta_thr = _meta_map[best_meta_name]

# ─────────────────────────────────────────────────────────────────────────────
# PART E — FINAL TEST EVALUATION with best meta-learner
# ─────────────────────────────────────────────────────────────────────────────
final_test_preds_meta = (best_meta_test_probs >= best_meta_thr).astype(int)
final_val_preds_meta  = (best_meta_val_probs  >= best_meta_thr).astype(int)

final_test_f1_meta  = f1_score(test_labels_emb,  final_test_preds_meta, zero_division=0)
final_test_auc_meta = roc_auc_score(test_labels_emb, best_meta_test_probs)

print("\n" + "=" * 55)
print(f"FINAL TEST ({best_meta_name})  F1={final_test_f1_meta:.4f}  AUC={final_test_auc_meta:.4f}")
print("=" * 55)
print(classification_report(test_labels_emb, final_test_preds_meta,
                             target_names=["Class 0", "Class 1"]))

# Confusion matrix
cm_meta = confusion_matrix(test_labels_emb, final_test_preds_meta)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_meta, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"])
plt.title(f"Confusion Matrix — {best_meta_name} (Cell 4 best meta)")
plt.tight_layout()
plt.show()

# McNemar test vs XGBoost baseline
xgb_test_preds_base = (xgb_test_probs >= xgb_best_thr).astype(int)
b_correct  = np.sum((xgb_test_preds_base == test_labels_emb) & (final_test_preds_meta == test_labels_emb))
xgb_only   = np.sum((xgb_test_preds_base == test_labels_emb) & (final_test_preds_meta != test_labels_emb))
best_only  = np.sum((xgb_test_preds_base != test_labels_emb) & (final_test_preds_meta == test_labels_emb))
b_wrong    = np.sum((xgb_test_preds_base != test_labels_emb) & (final_test_preds_meta != test_labels_emb))
cont_table = np.array([[b_correct, xgb_only], [best_only, b_wrong]])
mcn_result = mcnemar(cont_table, exact=True)
print(f"McNemar test (XGBoost baseline vs {best_meta_name}):")
print(f"  Contingency: {cont_table.tolist()}")
print(f"  p-value = {mcn_result.pvalue:.4f} "
      f"({'significant' if mcn_result.pvalue < 0.05 else 'not significant'} at α=0.05)")

# Feature importance (XGBoost or CatBoost only)
if best_meta_name in ("XGBoost", "CatBoost"):
    if best_meta_name == "XGBoost":
        importances = best_meta.feature_importances_
    else:
        importances = best_meta.get_feature_importance()
    top20_idx   = np.argsort(importances)[::-1][:20]
    top20_vals  = importances[top20_idx]
    top20_names = [f"{'EffNet' if i < 256 else 'ViT'}-{i % 256}" for i in top20_idx]
    colors_imp  = ["steelblue" if i < 256 else "coral" for i in top20_idx]
    plt.figure(figsize=(10, 5))
    plt.bar(range(20), top20_vals, color=colors_imp)
    plt.xticks(range(20), top20_names, rotation=45, ha="right", fontsize=8)
    plt.ylabel("Feature importance")
    plt.title(f"Top-20 Feature Importances — {best_meta_name}")
    blue_patch  = mpatches.Patch(color="steelblue", label="EfficientNet embedding")
    coral_patch = mpatches.Patch(color="coral",     label="ViT embedding")
    plt.legend(handles=[blue_patch, coral_patch])
    plt.tight_layout()
    plt.show()
else:
    print(f"Feature importance not available for {best_meta_name}.")

print(f"\nExpected F1 improvement after meta-learner benchmarking: ~{final_test_f1_meta:.3f}+")
free_memory()


## Cell 5 — Test Time Augmentation (TTA)

We apply **5 different views** of each image through the **best meta-learner** selected in Cell 4:

| View | Transform applied |
|------|------------------|
| 1 | Original (no augmentation) |
| 2 | Horizontal flip |
| 3 | Slight rotation (±10°) |
| 4 | Colour jitter |
| 5 | Centre crop (90 %) then resize |

Probabilities from all 5 views are **averaged** before thresholding.

In [ ]:
# ── 5.1  TTA transform list ───────────────────────────────────────────────────
tta_transforms = [
    # 1 — original
    T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    # 2 — horizontal flip
    T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=1.0),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    # 3 — slight rotation
    T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomRotation(degrees=10),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    # 4 — colour jitter
    T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    # 5 — centre crop
    T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.CenterCrop(int(IMG_SIZE * 0.9)),
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
]


# ── 5.2  Single-view embedding extraction ─────────────────────────────────────
@torch.no_grad()
def extract_embeddings_with_transform(model, dataset, transform, penultimate_layer):
    """
    Apply *transform* to PIL images from *dataset* and extract embeddings.
    *dataset* is expected to hold PIL images (transform=None).
    """
    model.eval()
    extractor = EmbeddingExtractor(model, penultimate_layer)
    all_embs, all_labels = [], []

    # We build a temporary loader with the desired transform
    tmp_ds = TransformSubset(dataset, transform)
    tmp_loader = DataLoader(
        tmp_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

    for imgs, labels in tmp_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            _ = model(imgs)
        all_embs.append(extractor.embedding.numpy())
        all_labels.extend(labels.numpy().tolist())

    extractor.remove()
    return np.vstack(all_embs), np.array(all_labels)


# ── 5.3  Run TTA over all 5 views on test set ─────────────────────────────────
# We work directly on the raw PIL dataset so each transform is applied fresh
raw_test_dataset = ImageFolder(root=str(TEST_DIR), transform=None)

# Validate set TTA (use the same val PIL images)
raw_val_pil = ImageFolder(root=str(TRAIN_DIR), transform=None)
raw_val_subset = Subset(raw_val_pil, val_idx)

tta_val_stack_probs_list  = []
tta_test_stack_probs_list = []

for i, tfm in enumerate(tta_transforms, 1):
    print(f"TTA view {i}/5…")

    # Val
    e_val, y_val = extract_embeddings_with_transform(
        eff_model, raw_val_subset, tfm, eff_penultimate)
    v_val, _     = extract_embeddings_with_transform(
        vit_model, raw_val_subset, tfm, vit_penultimate)
    X_v = feat_scaler.transform(np.hstack([e_val, v_val]))
    tta_val_stack_probs_list.append(best_meta.predict_proba(X_v)[:, 1])

    # Test
    e_tst, y_tst = extract_embeddings_with_transform(
        eff_model, raw_test_dataset, tfm, eff_penultimate)
    v_tst, _     = extract_embeddings_with_transform(
        vit_model, raw_test_dataset, tfm, vit_penultimate)
    X_t = feat_scaler.transform(np.hstack([e_tst, v_tst]))
    tta_test_stack_probs_list.append(best_meta.predict_proba(X_t)[:, 1])

# Average across views
tta_val_probs  = np.mean(tta_val_stack_probs_list,  axis=0)
tta_test_probs = np.mean(tta_test_stack_probs_list, axis=0)
tta_val_labels = y_val
tta_test_labels= y_tst

tta_val_preds = (tta_val_probs >= 0.5).astype(int)
tta_val_f1    = f1_score(tta_val_labels, tta_val_preds, zero_division=0)
print(f"\nTTA Ensemble val_F1={tta_val_f1:.4f}")
print(f"Expected F1 after TTA: ~{tta_val_f1:.3f}+")

free_memory()

## Cell 6 — Threshold Optimisation

The default 0.5 threshold is rarely optimal for imbalanced datasets.  
We **sweep thresholds from 0.20 to 0.80** and select the one that maximises F1 on the **validation set**.  
The optimised threshold is then applied to both validation and test predictions.

In [ ]:
# ── 6.1  Threshold sweep on validation set ────────────────────────────────────
thresholds   = np.linspace(0.20, 0.80, 61)
val_f1_scores = []

for thr in thresholds:
    preds = (tta_val_probs >= thr).astype(int)
    val_f1_scores.append(f1_score(tta_val_labels, preds, zero_division=0))

val_f1_scores   = np.array(val_f1_scores)
best_thr_idx    = np.argmax(val_f1_scores)
optimal_threshold = float(thresholds[best_thr_idx])
best_val_f1_opt   = float(val_f1_scores[best_thr_idx])

print(f"Optimal threshold : {optimal_threshold:.2f}")
print(f"Best val F1       : {best_val_f1_opt:.4f}")
print(f"Expected final F1 : ~{best_val_f1_opt:.3f}")

# ── 6.2  Plot threshold vs F1 ─────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(thresholds, val_f1_scores, color="steelblue", linewidth=2)
plt.axvline(optimal_threshold, color="red", linestyle="--",
            label=f"Optimal = {optimal_threshold:.2f}  (F1={best_val_f1_opt:.3f})")
plt.xlabel("Decision threshold")
plt.ylabel("Validation F1")
plt.title("Threshold vs Validation F1 (TTA stacking ensemble)")
plt.legend()
plt.tight_layout()
plt.show()

# ── 6.3  Apply optimal threshold to test ─────────────────────────────────────
final_test_preds = (tta_test_probs >= optimal_threshold).astype(int)
final_val_preds  = (tta_val_probs  >= optimal_threshold).astype(int)

## Cell 7 — Full Evaluation

Comprehensive evaluation of the **best meta-learner + TTA pipeline** covering:
- **Confusion matrix** and **classification report**
- **ROC curve** (AUC)
- **Precision-Recall curve** (AUC-PR)
- **McNemar's test** comparing EfficientNet standalone vs best stacking ensemble
- **Feature importance** from XGBoost or CatBoost if they win the benchmark
- **Error analysis** of false positives and false negatives

In [ ]:
# ── 7.1  Final test metrics ───────────────────────────────────────────────────
final_test_f1  = f1_score(tta_test_labels, final_test_preds, zero_division=0)
final_test_auc = roc_auc_score(tta_test_labels, tta_test_probs)
print("=" * 50)
print(f"FINAL TEST  F1  : {final_test_f1:.4f}")
print(f"FINAL TEST  AUC : {final_test_auc:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(tta_test_labels, final_test_preds,
                             target_names=["Class 0", "Class 1"]))

# ── 7.2  Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(tta_test_labels, final_test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix — Stacking Ensemble (TTA)")
plt.tight_layout()
plt.show()

# ── 7.3  ROC curve ────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(tta_test_labels, tta_test_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2,
         label=f"ROC (AUC = {final_test_auc:.3f})")
plt.plot([0, 1], [0, 1], color="navy", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Stacking Ensemble")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ── 7.4  Precision-Recall curve ───────────────────────────────────────────────
prec, rec, _ = precision_recall_curve(tta_test_labels, tta_test_probs)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec, color="green", lw=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Stacking Ensemble")
plt.tight_layout()
plt.show()

# ── 7.5  McNemar's test: EfficientNet vs Stacking Ensemble ────────────────────
# Use test-set predictions from EfficientNet (loaded checkpoint)
_, _, _, eff_test_probs_direct, eff_test_labels_direct = evaluate(eff_model, test_loader)
eff_test_preds_direct = (eff_test_probs_direct >= 0.5).astype(int)

# Build 2×2 contingency table
both_correct    = np.sum((eff_test_preds_direct == eff_test_labels_direct) &
                         (final_test_preds == tta_test_labels))
eff_only        = np.sum((eff_test_preds_direct == eff_test_labels_direct) &
                         (final_test_preds != tta_test_labels))
stack_only      = np.sum((eff_test_preds_direct != eff_test_labels_direct) &
                         (final_test_preds == tta_test_labels))
both_wrong      = np.sum((eff_test_preds_direct != eff_test_labels_direct) &
                         (final_test_preds != tta_test_labels))

contingency = np.array([[both_correct, eff_only],
                         [stack_only,  both_wrong]])
mcnemar_result = mcnemar(contingency, exact=True)
print("\nMcNemar's Test (EfficientNet vs Stacking Ensemble):")
print(f"  Contingency table:\n{contingency}")
print(f"  Statistic: {mcnemar_result.statistic:.4f}")
print(f"  p-value  : {mcnemar_result.pvalue:.4f}")
if mcnemar_result.pvalue < 0.05:
    print("  ✓ Significant improvement (p < 0.05)")
else:
    print("  ✗ Improvement not statistically significant (p ≥ 0.05)")

# ── 7.6  Feature importance (delegated to Cell 4 best meta-learner) ─────────────
# Feature importance plots and McNemar test were already generated in Cell 4.
# Here we reference the best meta-learner selected there for completeness.
print(f"Best meta-learner (Cell 4): {best_meta_name}")
print(f"  val_F1={benchmark_df.iloc[0]['val_F1']:.4f}  "  
      f"val_ROC_AUC={benchmark_df.iloc[0]['val_ROC_AUC']:.4f}  "
      f"val_PR_AUC={benchmark_df.iloc[0]['val_PR_AUC']:.4f}")

# ── 7.7  Error analysis: FP and FN ───────────────────────────────────────────
test_samples = test_dataset.samples
preds_arr  = final_test_preds
labels_arr = tta_test_labels

fp_idx = np.where((preds_arr == 1) & (labels_arr == 0))[0]
fn_idx = np.where((preds_arr == 0) & (labels_arr == 1))[0]

print(f"\nFalse Positives: {len(fp_idx)}")
print(f"False Negatives: {len(fn_idx)}")

def show_error_images(indices, title, n=8):
    if len(indices) == 0:
        print(f"No {title} samples.")
        return
    indices = indices[:n]
    fig, axes = plt.subplots(1, len(indices), figsize=(2 * len(indices), 2.5))
    if len(indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, indices):
        img_path = test_samples[idx][0]
        from PIL import Image
        img = Image.open(img_path).convert("RGB").resize((112, 112))
        ax.imshow(img)
        ax.set_title(f"p={tta_test_probs[idx]:.2f}")
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_error_images(fp_idx, "False Positives (predicted 1, true 0)")
show_error_images(fn_idx, "False Negatives (predicted 0, true 1)")

free_memory()

## Cell 8 — Interpretability: Grad-CAM

**Gradient-weighted Class Activation Mapping (Grad-CAM)** highlights the regions in each image that drove the EfficientNet prediction.  

We use the `pytorch-grad-cam` library to generate CAM overlays for:
- **Correct predictions** (true positives + true negatives)
- **Incorrect predictions** (false positives + false negatives)

The target layer is the **last convolutional layer** of EfficientNet-B0 (`features[-1][0]`).

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
from PIL import Image
import numpy as np

# ── 8.1  Grad-CAM setup ───────────────────────────────────────────────────────
# Reload best EfficientNet weights (just in case)
eff_model.load_state_dict(torch.load("best_efficientnet.pth", map_location=DEVICE))
eff_model.eval()

# Target layer: last Conv block in EfficientNet-B0 features
target_layers = [eff_model.features[-1][0]]

# Wrapper to make EfficientNet output a scalar (required by pytorch-grad-cam)
class EfficientNetWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x).squeeze(1)   # (B, 1) → (B,)

wrapped_model = EfficientNetWrapper(eff_model)

cam_engine = GradCAM(
    model=wrapped_model,
    target_layers=target_layers,
)


# ── 8.2  Helper: generate Grad-CAM overlay ────────────────────────────────────
def get_gradcam(image_path: str, label: int):
    """
    Returns the CAM overlay (H×W×3 float32 in [0,1])
    alongside the original RGB image.
    """
    pil_img = Image.open(image_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    rgb_np  = np.array(pil_img, dtype=np.float32) / 255.0    # (H, W, 3)

    tensor  = val_transform(pil_img).unsqueeze(0).to(DEVICE)  # (1, 3, H, W)

    targets = [BinaryClassifierOutputTarget(label)]
    grayscale_cam = cam_engine(input_tensor=tensor, targets=targets)[0]  # (H, W)

    overlay = show_cam_on_image(rgb_np, grayscale_cam, use_rgb=True)
    return overlay, pil_img


# ── 8.3  Visualise correct and incorrect predictions ─────────────────────────
def visualise_gradcam(indices, title, n=4):
    indices = indices[:n]
    if len(indices) == 0:
        print(f"No samples for: {title}")
        return
    fig, axes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 6))
    if len(indices) == 1:
        axes = axes.reshape(2, 1)
    for col, idx in enumerate(indices):
        img_path  = test_samples[idx][0]
        true_lbl  = int(tta_test_labels[idx])
        pred_lbl  = int(final_test_preds[idx])
        prob_val  = float(tta_test_probs[idx])

        overlay, orig = get_gradcam(img_path, true_lbl)

        axes[0, col].imshow(orig)
        axes[0, col].set_title(f"True:{true_lbl}  Pred:{pred_lbl}\np={prob_val:.2f}",
                               fontsize=9)
        axes[0, col].axis("off")

        axes[1, col].imshow(overlay)
        axes[1, col].set_title("Grad-CAM", fontsize=9)
        axes[1, col].axis("off")

    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Correct predictions
tp_idx = np.where((final_test_preds == 1) & (tta_test_labels == 1))[0]
tn_idx = np.where((final_test_preds == 0) & (tta_test_labels == 0))[0]

print("Grad-CAM: True Positives")
visualise_gradcam(tp_idx, "Grad-CAM — True Positives (correct)")

print("Grad-CAM: True Negatives")
visualise_gradcam(tn_idx, "Grad-CAM — True Negatives (correct)")

print("Grad-CAM: False Positives")
visualise_gradcam(fp_idx, "Grad-CAM — False Positives (predicted 1, true 0)")

print("Grad-CAM: False Negatives")
visualise_gradcam(fn_idx, "Grad-CAM — False Negatives (predicted 0, true 1)")

free_memory()
print("\n✓ All cells complete. Notebook is submission-ready.")